<a href="https://colab.research.google.com/github/AzizulHakim00/Glaucomma/blob/main/Glaucomma_RimGraphDG_V45_ORIGA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RimGraph-DG V4.5 — mask-audited ORIGA-first run
Use a **T4 GPU**. This run validates decoded OD/OC supervision before model training, reuses the completed V4.4 ORIGA baseline only if split/config signatures match exactly, and trains RimGraph fresh.

In [ ]:
GLAUCOMMA_OVERRIDES = {
    'manual_data_dir': '',
    'fast_dev_run': False,
    'run_name': 'paper_run_v45',
    'code_revision': 'rimgraph-dg-v4.5-20260809',
    'seeds': [2029],
    'fold_targets': ['ORIGA'],
    'run_global_baseline': True,
    'run_full_model': True,
    'baseline_reuse_run': 'paper_run_v44',
    'run_optuna': False,
    'image_size': 320,
    'batch_size': 2,
    'grad_accum': 2,
    'fpn_dim': 128,
    'gradient_checkpointing': True,
    'baseline_epochs': 12,
    'full_epochs': 30,
    'seg_warmup_epochs': 5,
    'anatomy_warmup_epochs': 10,
    'num_workers': 0,
    'n_visual_examples': 2,
    'resume': True,
}

import hashlib, json, traceback, urllib.request
from pathlib import Path
import torch

print('=== V4.5 LAUNCHER GPU CHECK ===', flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('CUDA available:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('T4 GPU is not active. Colab: Runtime > Change runtime type > T4 GPU, reconnect, then rerun.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('================================', flush=True)

COMMIT = 'd71c11be1a13a25ed0352cdde01453c17b234429'
EXPECTED_RAW_SHA256 = '46ba27c7446662460456bc2bab186729c0df1b3e76533ce44f208150208335e2'
ROOT = f'https://raw.githubusercontent.com/AzizulHakim00/Glaucomma/{COMMIT}'
parts = [f'v4_parts/part_{i:02d}.py' for i in range(7)]
raw_code = '\n'.join(urllib.request.urlopen(f'{ROOT}/{name}').read().decode('utf-8') for name in parts)
actual_raw = hashlib.sha256(raw_code.encode('utf-8')).hexdigest()
assert actual_raw == EXPECTED_RAW_SHA256, f'Raw runner integrity check failed: {actual_raw}'

patch_specs = [
    ('runner_patch_v41.py', 'apply_v41'),
    ('runner_patch_v42.py', 'apply_v42'),
    ('runner_patch_v43.py', 'apply_v43'),
    ('runner_patch_v43_autograd.py', 'apply_v43_autograd'),
    ('runner_patch_v44_runtime.py', 'apply_v44_runtime'),
    ('runner_patch_v45_masks.py', 'apply_v45_masks'),
    ('runner_patch_v45_lowlabels.py', 'apply_v45_lowlabels'),
]
code = raw_code
for patch_name, function_name in patch_specs:
    print(f'[LAUNCHER] applying {patch_name}', flush=True)
    source = urllib.request.urlopen(f'{ROOT}/{patch_name}').read().decode('utf-8')
    namespace = {}
    exec(compile(source, patch_name, 'exec'), namespace, namespace)
    code = namespace[function_name](code)
compile(code, 'rimgraph_dg_v45_origa.py', 'exec')
print('[LAUNCHER] V4.5 assembly PASSED', flush=True)

try:
    exec(code, globals(), globals())
    completion = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45/RUN_COMPLETED.json')
    if not completion.exists():
        raise RuntimeError('Runner returned without RUN_COMPLETED.json; treating this as failure.')
    print('\n✅ V4.5 VERIFIED COMPLETION:', completion, flush=True)
except BaseException:
    trace = traceback.format_exc()
    print('\n=== V4.5 FAILURE TRACEBACK ===', flush=True)
    print(trace, flush=True)
    try:
        Path('/content/RimGraph_V45_FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
    except Exception:
        pass
    try:
        failure_dir = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45')
        failure_dir.mkdir(parents=True, exist_ok=True)
        (failure_dir / 'FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
        (failure_dir / 'FAILURE_STATUS.json').write_text(json.dumps({'status': 'failed', 'traceback_file': str(failure_dir / 'FAILURE_TRACEBACK.txt')}, indent=2), encoding='utf-8')
    except Exception as drive_error:
        print(f'Could not persist failure to Drive: {drive_error}', flush=True)
    raise


In [ ]:

# ================================================================
# GLAUCOMMA — RIMGRAPH-DG V4.5 A-G ABLATION EXPERIMENT
# Single-cell | Google Drive | Resumable | T4 GPU
#
# A: Global ConvNeXt baseline
# B: Global + segmentation
# C: B + ROI + vCDR
# D: C + 12-sector structural features
# E: D + graph attention
# F: E + domain adversarial learning
# G: F + prototype alignment
#
# Based on the pinned V4.5 repository implementation.
# ================================================================

# ======================== 1. USER SETTINGS ========================

RUN_MODE = "pilot"   # "pilot" or "full"

# pilot = ORIGA, seed 2029, all A-G
# full  = ORIGA + REFUGE + G1020, seeds 2029/2030/2031

RUN_VARIANT_IDS = ["B", "C", "D", "E", "F", "G"]

IMAGE_SIZE = 320
BATCH_SIZE = 2
GRAD_ACCUM = 2

BASELINE_EPOCHS = 12
FULL_EPOCHS = 30
PATIENCE = 8

BOOTSTRAP_SAMPLES = 500

RESUME = True
SAVE_EVERY_EPOCH = True

# Save model weights as .pkl for A and G by default.
# Set True to export .pkl for every model (requires more Drive space).
SAVE_PICKLE_FOR_ALL_VARIANTS = False
PICKLE_VARIANTS = {"global_baseline", "G_prototype_alignment"}

# Reuse the existing V4.5 ORIGA baseline only if its split and
# baseline-sensitive configuration match exactly.
BASELINE_REUSE_RUN = "paper_run_v45"

# Increment this whenever you intentionally change the experiment
# implementation or make a scientific configuration change.
EXPERIMENT_VERSION = "ablation_v1"

if RUN_MODE == "pilot":
    FOLD_TARGETS = ["ORIGA"]
    SEEDS = [2029]
elif RUN_MODE == "full":
    FOLD_TARGETS = ["ORIGA", "REFUGE", "G1020"]
    SEEDS = [2029, 2030, 2031]
else:
    raise ValueError("RUN_MODE must be 'pilot' or 'full'.")

# ======================== 2. EXPERIMENT MATRIX ========================

ABLATION_PLAN = [
    {
        "id": "B",
        "name": "B_global_segmentation",
        "stage": "global_seg",
        "domain": False,
        "prototype": False,
        "description": "Global classification with disc/cup segmentation supervision"
    },
    {
        "id": "C",
        "name": "C_roi_vcdr",
        "stage": "vcdr_only",
        "domain": False,
        "prototype": False,
        "description": "ROI features and vCDR anatomical information"
    },
    {
        "id": "D",
        "name": "D_sector_features",
        "stage": "sector_vcdr",
        "domain": False,
        "prototype": False,
        "description": "ROI, vCDR and sector spectral features without GAT"
    },
    {
        "id": "E",
        "name": "E_graph_attention",
        "stage": "full",
        "domain": False,
        "prototype": False,
        "description": "Full anatomical fusion with graph attention"
    },
    {
        "id": "F",
        "name": "F_domain_adversarial",
        "stage": "full",
        "domain": True,
        "prototype": False,
        "description": "Graph attention plus domain adversarial learning"
    },
    {
        "id": "G",
        "name": "G_prototype_alignment",
        "stage": "full",
        "domain": True,
        "prototype": True,
        "description": "Complete RimGraph-DG with prototype alignment"
    }
]

MODEL_ORDER = [
    "global_baseline",
    "B_global_segmentation",
    "C_roi_vcdr",
    "D_sector_features",
    "E_graph_attention",
    "F_domain_adversarial",
    "G_prototype_alignment"
]

# ======================== 3. V4.5 CONFIGURATION ========================

GLAUCOMMA_OVERRIDES = {
    "manual_data_dir": "",
    "kaggle_dataset": "arnavjain1/glaucoma-datasets",

    "sources": ["ORIGA", "REFUGE", "G1020"],
    "fold_targets": FOLD_TARGETS,
    "seeds": SEEDS,

    "run_name": "rimgraph_ablation_v1",
    "code_revision": "rimgraph-dg-v45-ablation-20261113",

    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,

    "baseline_epochs": BASELINE_EPOCHS,
    "full_epochs": FULL_EPOCHS,
    "seg_warmup_epochs": 5,
    "anatomy_warmup_epochs": 10,
    "patience": PATIENCE,

    "fpn_dim": 128,
    "feature_dim": 256,
    "graph_hidden": 128,
    "graph_heads": 4,
    "num_sectors": 12,
    "roi_output_size": 20,

    "backbone": "convnext_tiny.fb_in22k_ft_in1k",
    "backbone_fallback": "convnext_tiny",
    "pretrained": True,
    "gradient_checkpointing": True,

    "lr": 2e-4,
    "backbone_lr_scale": 0.10,
    "weight_decay": 1e-4,
    "dropout": 0.25,

    "lambda_seg": 1.0,
    "lambda_vcdr": 0.20,
    "lambda_cons": 0.10,
    "lambda_domain": 0.04,
    "lambda_proto": 0.04,

    "val_fraction": 0.20,
    "mixed_precision": True,
    "num_workers": 0,

    "run_global_baseline": True,
    "run_full_model": False,
    "run_optuna": False,

    "fast_dev_run": False,
    "resume": RESUME,
    "save_every_epoch": SAVE_EVERY_EPOCH,

    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "n_visual_examples": 4,

    "exclude_cross_source_duplicates": True,
    "canonicalize_laterality": True,

    "baseline_reuse_run": BASELINE_REUSE_RUN,
    "drive_folder": "Glaucomma_RimGraphDG"
}

# ======================== 4. DOWNLOAD PINNED V4.5 SOURCE ========================

import hashlib
import json
import urllib.request
import traceback

REPO = "https://raw.githubusercontent.com/AzizulHakim00/Glaucomma"
COMMIT = "d71c11be1a13a25ed0352cdde01453c17b234429"

EXPECTED_RAW_SHA256 = (
    "46ba27c7446662460456bc2bab186729"
    "c0df1b3e76533ce44f208150208335e2"
)

PARTS = [f"v4_parts/part_{i:02d}.py" for i in range(7)]

PATCHES = [
    ("runner_patch_v41.py", "apply_v41"),
    ("runner_patch_v42.py", "apply_v42"),
    ("runner_patch_v43.py", "apply_v43"),
    ("runner_patch_v43_autograd.py", "apply_v43_autograd"),
    ("runner_patch_v44_runtime.py", "apply_v44_runtime"),
    ("runner_patch_v45_masks.py", "apply_v45_masks"),
    ("runner_patch_v45_lowlabels.py", "apply_v45_lowlabels"),
]

print("Downloading pinned Glaucomma V4.5 implementation...")

raw_code = "\n".join(
    urllib.request.urlopen(
        f"{REPO}/{COMMIT}/{name}", timeout=60
    ).read().decode("utf-8")
    for name in PARTS
)

actual_hash = hashlib.sha256(raw_code.encode("utf-8")).hexdigest()

assert actual_hash == EXPECTED_RAW_SHA256, (
    "Source integrity verification failed. "
    f"Expected {EXPECTED_RAW_SHA256}, got {actual_hash}"
)

code = raw_code

for patch_name, function_name in PATCHES:
    print("Applying:", patch_name)

    source = urllib.request.urlopen(
        f"{REPO}/{COMMIT}/{patch_name}", timeout=60
    ).read().decode("utf-8")

    namespace = {}
    exec(compile(source, patch_name, "exec"), namespace, namespace)
    code = namespace[function_name](code)

print("V4.5 source integrity and original patches: PASSED")

# ======================== 5. SAFE CODE MODIFICATIONS ========================

def replace_once(source, old, new, label):
    count = source.count(old)
    if count != 1:
        raise RuntimeError(
            f"Patch '{label}' expected one match, found {count}. "
            "The pinned implementation may have changed."
        )
    return source.replace(old, new, 1)


def replace_all_checked(source, old, new, expected, label):
    count = source.count(old)
    if count != expected:
        raise RuntimeError(
            f"Patch '{label}' expected {expected} matches, found {count}."
        )
    return source.replace(old, new)


# 5.1 Separate vCDR and sector feature ablations.
# V4.3 stores 5 global anatomical values followed by the
# Fourier spectrum of the 12-sector rim vector.

old_model = '''        fs = self.struct_proj(struct)
        q = torch.sigmoid(self.quality_head(global_raw))
        if stage == "global_seg":
            gates = torch.zeros(x.shape[0], 4, device=x.device, dtype=x.dtype); gates[:, 0] = 1.0
        else:
            gate_logits = self.gate(torch.cat([fg, fl, fr, fs, q], dim=1))
            if stage == "anatomy": gate_logits[:, 2] = -1e4
            gates = torch.softmax(gate_logits, dim=1)
'''

new_model = '''        # Controlled anatomical ablations.
        # Index 3 = vCDR; indices 5 onward = sector spectrum.
        if stage in {"vcdr_only", "sector_vcdr"}:
            selected_struct = torch.zeros_like(struct)
            selected_struct[:, 3] = struct[:, 3]

            if stage == "sector_vcdr":
                selected_struct[:, 5:] = struct[:, 5:]

            struct = selected_struct

            # Important: prevent the disabled graph from indirectly
            # influencing the remaining branches through the gate network.
            fr = torch.zeros_like(fr)

        fs = self.struct_proj(struct)
        q = torch.sigmoid(self.quality_head(global_raw))

        if stage == "global_seg":
            gates = torch.zeros(x.shape[0], 4, device=x.device, dtype=x.dtype)
            gates[:, 0] = 1.0
        else:
            gate_logits = self.gate(torch.cat([fg, fl, fr, fs, q], dim=1))
            if stage in {"anatomy", "vcdr_only", "sector_vcdr"}:
                gate_logits[:, 2] = -1e4
            gates = torch.softmax(gate_logits, dim=1)
'''

code = replace_once(code, old_model, new_model, "anatomical feature isolation")


# 5.2 Static stages for B/C/D.
# E/F/G retain the original staged training schedule.

code = replace_once(
    code,
    "    stage = full_stage(epoch)\n",
    '''    fixed_stage = CFG.get("ablation_stage", "full")
    if fixed_stage in {"global_seg", "vcdr_only", "sector_vcdr"}:
        stage = fixed_stage
    else:
        stage = full_stage(epoch)
''',
    "fixed ablation stages"
)


# 5.3 Activate vCDR supervision for C and D.

code = replace_once(
    code,
    '            if stage in {"anatomy", "full"}:\n',
    '            if stage in {"anatomy", "full", "vcdr_only", "sector_vcdr"}:\n',
    "vCDR loss stages"
)


# 5.4 Independently enable domain and prototype objectives.

old_domain_block = '''            if stage == "full":
                l_domain = F.cross_entropy(out["domain"], batch["domain"])
                pair_count = prototype_pair_count(batch["label"], batch["domain"])
                l_proto = prototype_loss(out["fused"], batch["label"], batch["domain"])
                full_stage_batches += 1; prototype_pairs += int(pair_count); prototype_active_batches += int(pair_count > 0)
                loss = loss + CFG["lambda_domain"] * l_domain + CFG["lambda_proto"] * l_proto
'''

new_domain_block = '''            if stage == "full":
                full_stage_batches += 1

                if CFG.get("use_domain_loss", False):
                    l_domain = F.cross_entropy(out["domain"], batch["domain"])
                    loss = loss + CFG["lambda_domain"] * l_domain

                if CFG.get("use_prototype_loss", False):
                    pair_count = prototype_pair_count(
                        batch["label"], batch["domain"]
                    )
                    l_proto = prototype_loss(
                        out["fused"], batch["label"], batch["domain"]
                    )
                    prototype_pairs += int(pair_count)
                    prototype_active_batches += int(pair_count > 0)
                    loss = loss + CFG["lambda_proto"] * l_proto
'''

code = replace_once(
    code, old_domain_block, new_domain_block,
    "independent domain and prototype objectives"
)


# 5.5 Test-time feature stage must match the trained ablation.

code = replace_once(
    code,
    'out = model(batch["image"], stage="full", grl_lambda=0.0)',
    'out = model(batch["image"], stage=CFG.get("ablation_stage", "full"), grl_lambda=0.0)',
    "ablation inference stage"
)


# 5.6 Cup-mask guard for the new anatomical ablation stages.

code = replace_once(
    code,
    'if train and stage in {"anatomy", "full"} and int(metrics.get("n_mask_cup", 0)) == 0:',
    'if train and stage in {"anatomy", "full", "vcdr_only", "sector_vcdr"} and int(metrics.get("n_mask_cup", 0)) == 0:',
    "cup supervision guard"
)


# 5.7 Checkpoint compatibility must include the experiment signature.

old_compat = (
    'return cfg.get("code_revision") == CFG["code_revision"] '
    'and checkpoint.get("model_type") == model_type '
    'and int(checkpoint.get("seed", -1)) == int(seed)'
)

new_compat = (
    'return cfg.get("code_revision") == CFG["code_revision"] '
    'and checkpoint.get("model_type") == model_type '
    'and int(checkpoint.get("seed", -1)) == int(seed) '
    'and cfg.get("experiment_signature") == CFG.get("experiment_signature")'
)

code = replace_once(
    code, old_compat, new_compat,
    "checkpoint experiment signature"
)


# Completed experiments are reused only with matching signatures.

code = replace_all_checked(
    code,
    'if saved.get("code_revision") == CFG["code_revision"]:\n',
    'if saved.get("code_revision") == CFG["code_revision"] and saved.get("experiment_signature") == CFG.get("experiment_signature"):\n',
    2,
    "completed-run signature"
)

code = replace_all_checked(
    code,
    'STORE.save_json({"metrics": metrics, "code_revision": CFG["code_revision"]}, f"{rel}/COMPLETED.json")',
    'STORE.save_json({"metrics": metrics, "code_revision": CFG["code_revision"], "experiment_signature": CFG.get("experiment_signature")}, f"{rel}/COMPLETED.json")',
    2,
    "completion signature"
)


# 5.8 Make train_full use a separate checkpoint directory per ablation.

train_start = "def train_full(target_source, seed, train_df, val_df, test_df, root_rel):\n"
train_end = "\n\n# ---------------------------- 6."

a = code.index(train_start)
b = code.index(train_end, a)

train_block = code[a:b]

def block_replace(block, old, new, label):
    count = block.count(old)
    if count != 1:
        raise RuntimeError(f"Training patch {label}: expected 1, found {count}")
    return block.replace(old, new, 1)

train_block = block_replace(
    train_block,
    'rel = f"{root_rel}/rimgraph_v4"; STORE.dirs(rel)',
    'rel = f"{root_rel}/{CFG["ablation_name"]}"; STORE.dirs(rel)',
    "unique model directory"
)

train_block = block_replace(
    train_block,
    'checkpoint_compatible(ck, "rimgraph_v4", seed)',
    'checkpoint_compatible(ck, CFG["ablation_name"], seed)',
    "model checkpoint identity"
)

train_block = block_replace(
    train_block,
    '"model_type": "rimgraph_v4"',
    '"model_type": CFG["ablation_name"]',
    "checkpoint model identity"
)

train_block = block_replace(
    train_block,
    'finalize_predictions(val_pred, test_pred, target_source, "rimgraph_v4", seed, rel, seg_metrics)',
    'finalize_predictions(val_pred, test_pred, target_source, CFG["ablation_name"], seed, rel, seg_metrics)',
    "prediction model identity"
)

train_block = block_replace(
    train_block,
    'score = 0.65 * np.nan_to_num(va["auroc"]) + 0.20 * np.nan_to_num(va["auprc"]) + 0.075 * np.nan_to_num(va["dice_disc"]) + 0.075 * np.nan_to_num(va["dice_cup"])',
    'score = 0.70 * np.nan_to_num(va["auroc"]) + 0.30 * np.nan_to_num(va["auprc"])',
    "consistent classification checkpoint selection"
)

train_block = block_replace(
    train_block,
    'eligible_for_selection = va["stage"] == "full"',
    'eligible_for_selection = (va["stage"] == "full") or (CFG["ablation_stage"] != "full")',
    "ablation checkpoint selection"
)

train_block = block_replace(
    train_block,
    'if va["stage"] == "full" and epoch - best_epoch >= CFG["patience"]: break',
    'if (va["stage"] == "full" or CFG["ablation_stage"] != "full") and epoch - best_epoch >= CFG["patience"] and epoch > CFG["anatomy_warmup_epochs"]: break',
    "ablation early stopping"
)

code = code[:a] + train_block + code[b:]


# ======================== 6. REPLACE ORIGINAL MAIN LOOP ========================

main_marker = (
    'for seed in CFG["seeds"]:\n'
    '    seed_everything(int(seed))\n'
    '    for target in CFG["fold_targets"]:\n'
)

if code.count(main_marker) != 1:
    raise RuntimeError("Could not locate the original V4.5 main experiment loop.")

custom_main = r'''
# ================================================================
# CUSTOM A-G ABLATION EXECUTION
# ================================================================

import pickle
import time
import gc

ALL_METRICS, ALL_PREDS = [], []

BASE_REVISION = CFG["code_revision"]

SIGNATURE_KEYS = [
    "image_size", "batch_size", "grad_accum",
    "backbone", "backbone_fallback", "pretrained",
    "baseline_epochs", "full_epochs", "lr",
    "backbone_lr_scale", "weight_decay", "focal_gamma",
    "lambda_seg", "lambda_vcdr", "lambda_cons",
    "lambda_domain", "lambda_proto",
    "val_fraction", "num_sectors", "roi_output_size",
    "fpn_dim", "feature_dim", "graph_hidden",
    "graph_heads", "dropout", "patience",
    "canonicalize_laterality",
    "exclude_cross_source_duplicates",
]


def set_experiment_signature(name, stage, domain, prototype, seed, target):
    payload = {
        "version": EXPERIMENT_VERSION,
        "name": name,
        "stage": stage,
        "domain": domain,
        "prototype": prototype,
        "seed": int(seed),
        "target": target,
        "fold_targets": CFG["fold_targets"],
        "settings": {k: CFG.get(k) for k in SIGNATURE_KEYS},
    }

    signature = hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode()
    ).hexdigest()[:16]

    CFG["experiment_signature"] = signature
    CFG["code_revision"] = f"{BASE_REVISION}-{name}-{signature}"

    return signature


def specificity_at_90_sensitivity(y, p):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    fpr, tpr, _ = roc_curve(y, p)
    valid = np.where(tpr >= 0.90)[0]

    return float(1.0 - fpr[valid].min()) if len(valid) else 0.0


def export_checkpoint_artifacts(rel, architecture, model_name, metrics):
    checkpoint_path = find_existing(f"{rel}/best_model.pt")

    if checkpoint_path is None:
        raise FileNotFoundError(
            f"Best checkpoint not found: {rel}/best_model.pt"
        )

    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False
    )

    state = checkpoint["model"]

    parameter_count = sum(
        int(v.numel()) for v in state.values()
        if torch.is_tensor(v)
    )

    # Standard PyTorch state dictionary.
    local_weights = LOCAL_RUN / rel / "model_state_dict.pt"
    local_weights.parent.mkdir(parents=True, exist_ok=True)

    torch.save(state, local_weights)
    STORE.mirror(local_weights, f"{rel}/model_state_dict.pt")

    # Optional portable NumPy-based pickle.
    if SAVE_PICKLE_FOR_ALL_VARIANTS or model_name in PICKLE_VARIANTS:
        portable_state = {
            k: v.detach().cpu().numpy() if torch.is_tensor(v) else v
            for k, v in state.items()
        }

        local_pkl = LOCAL_RUN / rel / "model.pkl"

        with open(local_pkl, "wb") as f:
            pickle.dump(
                {
                    "architecture": architecture,
                    "model_name": model_name,
                    "experiment_signature": CFG["experiment_signature"],
                    "config": dict(CFG),
                    "state_dict": portable_state,
                },
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

        STORE.mirror(local_pkl, f"{rel}/model.pkl")

    STORE.save_json(
        {
            "architecture": architecture,
            "model_name": model_name,
            "parameters": parameter_count,
            "experiment_signature": CFG["experiment_signature"],
            "checkpoint": f"{rel}/best_model.pt",
            "state_dict": f"{rel}/model_state_dict.pt",
        },
        f"{rel}/model_info.json"
    )

    metrics["model_parameters"] = parameter_count

    return metrics


def finalize_metric_record(metrics, predictions, rel):
    p = predictions["prob_calibrated"].values
    y = predictions["label"].values

    metrics["specificity_at_90_sensitivity"] = (
        specificity_at_90_sensitivity(y, p)
    )

    metrics["n_test_images"] = int(len(predictions))

    STORE.save_json(metrics, f"{rel}/metrics.json")

    completed_path = find_existing(f"{rel}/COMPLETED.json")

    if completed_path is not None:
        completed = json.loads(
            completed_path.read_text(encoding="utf-8")
        )

        completed["metrics"] = metrics
        completed["code_revision"] = CFG["code_revision"]
        completed["experiment_signature"] = CFG["experiment_signature"]

        STORE.save_json(completed, f"{rel}/COMPLETED.json")

    return metrics


def save_prediction_artifacts(pred, target, seed, model_name, ablation_id, rel):
    pred = pred.copy()

    pred["held_out_source"] = target
    pred["seed"] = int(seed)
    pred["model_type"] = model_name
    pred["ablation_id"] = ablation_id

    STORE.save_df(pred, f"{rel}/test_predictions.csv")

    return pred


def save_model_figures(pred, metrics, rel, title):
    figure_rel = f"{rel}/figures"
    STORE.dirs(figure_rel)

    history_path = find_existing(f"{rel}/history.csv")

    if history_path is not None:
        save_training_curves(
            history_path, figure_rel, title
        )

    save_discrimination_figure(
        pred, metrics, figure_rel, title
    )


def load_full_model_from_checkpoint(rel, train_df):
    domain_map = {
        s: i for i, s in enumerate(sorted(train_df.source.unique()))
    }

    checkpoint_path = find_existing(f"{rel}/best_model.pt")

    checkpoint = torch.load(
        checkpoint_path, map_location=DEVICE, weights_only=False
    )

    model = RimGraphV4(num_domains=len(domain_map)).to(DEVICE)
    model.load_state_dict(checkpoint["model"])

    return model


def paired_bootstrap_auc_delta(
    pred_a, pred_b, seed=2029, n_boot=500
):
    a = pred_a[["path", "label", "prob_calibrated"]].copy()
    b = pred_b[["path", "label", "prob_calibrated"]].copy()

    joined = a.merge(
        b,
        on=["path", "label"],
        how="inner",
        suffixes=("_a", "_b"),
        validate="one_to_one"
    )

    y = joined["label"].values.astype(int)
    pa = joined["prob_calibrated_a"].values
    pb = joined["prob_calibrated_b"].values

    if len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan, len(joined)

    delta = roc_auc_score(y, pb) - roc_auc_score(y, pa)

    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]

    rng = np.random.default_rng(seed)
    values = []

    for _ in range(int(n_boot)):
        idx = np.concatenate([
            rng.choice(pos, len(pos), replace=True),
            rng.choice(neg, len(neg), replace=True)
        ])

        try:
            values.append(
                roc_auc_score(y[idx], pb[idx])
                - roc_auc_score(y[idx], pa[idx])
            )
        except Exception:
            continue

    if not values:
        return delta, np.nan, np.nan, len(joined)

    return (
        float(delta),
        float(np.percentile(values, 2.5)),
        float(np.percentile(values, 97.5)),
        len(joined)
    )


print("\n" + "=" * 65)
print("GLAUCOMMA A-G ABLATION EXPERIMENT")
print("Run mode:", RUN_MODE)
print("Folds:", CFG["fold_targets"])
print("Seeds:", CFG["seeds"])
print("Resume:", CFG["resume"])
print("=" * 65)

STORE.save_df(
    pd.DataFrame(ABLATION_PLAN),
    "ablation_experiment_plan.csv"
)

# ------------------------ MAIN EXPERIMENT LOOP ------------------------

for seed in CFG["seeds"]:

    seed_everything(int(seed))

    for target in CFG["fold_targets"]:

        print("\n" + "#" * 65)
        print(f"SEED {seed} | HELD-OUT DATASET: {target}")
        print("#" * 65)

        root_rel = f"folds/{target}/seed_{seed}"

        train_df, val_df, test_df = prepare_fold_data(
            target, int(seed), root_rel
        )

        # ============================================================
        # A. GLOBAL CONVNEXT BASELINE
        # ============================================================

        baseline_name = "global_baseline"

        signature = set_experiment_signature(
            baseline_name,
            "global",
            False,
            False,
            seed,
            target
        )

        CFG["ablation_name"] = baseline_name
        CFG["ablation_stage"] = "global_seg"
        CFG["use_domain_loss"] = False
        CFG["use_prototype_loss"] = False

        baseline_rel = f"{root_rel}/global_baseline"

        print("\n[A] Training or resuming global baseline...")

        baseline_metrics, baseline_pred = train_baseline(
            target, int(seed), train_df, val_df, test_df, root_rel
        )

        baseline_pred = save_prediction_artifacts(
            baseline_pred,
            target,
            seed,
            baseline_name,
            "A",
            baseline_rel
        )

        baseline_metrics = export_checkpoint_artifacts(
            baseline_rel,
            "GlobalBaseline",
            baseline_name,
            baseline_metrics
        )

        baseline_metrics = finalize_metric_record(
            baseline_metrics,
            baseline_pred,
            baseline_rel
        )

        STORE.save_df(
            baseline_pred,
            f"{baseline_rel}/test_predictions.csv"
        )

        save_model_figures(
            baseline_pred,
            baseline_metrics,
            baseline_rel,
            f"A — Global ConvNeXt — {target}"
        )

        ALL_METRICS.append(baseline_metrics)
        ALL_PREDS.append(baseline_pred)

        del baseline_pred
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # ============================================================
        # B-G. CONTROLLED ABLATION EXPERIMENTS
        # ============================================================

        for spec in ABLATION_PLAN:

            if spec["id"] not in RUN_VARIANT_IDS:
                continue

            name = spec["name"]
            stage = spec["stage"]

            signature = set_experiment_signature(
                name,
                stage,
                spec["domain"],
                spec["prototype"],
                seed,
                target
            )

            CFG["ablation_name"] = name
            CFG["ablation_stage"] = stage
            CFG["use_domain_loss"] = spec["domain"]
            CFG["use_prototype_loss"] = spec["prototype"]

            variant_rel = f"{root_rel}/{name}"

            print("\n" + "-" * 60)
            print(f"[{spec['id']}] {name}")
            print(spec["description"])
            print("Stage:", stage)
            print("Domain objective:", spec["domain"])
            print("Prototype objective:", spec["prototype"])
            print("Signature:", signature)
            print("-" * 60)

            metrics, pred, model = train_full(
                target,
                int(seed),
                train_df,
                val_df,
                test_df,
                root_rel
            )

            pred = save_prediction_artifacts(
                pred,
                target,
                seed,
                name,
                spec["id"],
                variant_rel
            )

            metrics = export_checkpoint_artifacts(
                variant_rel,
                "RimGraphV4",
                name,
                metrics
            )

            metrics = finalize_metric_record(
                metrics,
                pred,
                variant_rel
            )

            STORE.save_df(
                pred,
                f"{variant_rel}/test_predictions.csv"
            )

            save_model_figures(
                pred,
                metrics,
                variant_rel,
                f"{spec['id']} — {name} — {target}"
            )

            # Full-model qualitative visualizations.
            if spec["id"] == "G":

                if model is None:
                    model = load_full_model_from_checkpoint(
                        variant_rel, train_df
                    )

                print("[G] Saving anatomical and Grad-CAM visualizations...")

                save_full_xai(
                    model,
                    test_df,
                    int(seed),
                    f"{variant_rel}/figures"
                )

                model.to("cpu")
                del model

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            ALL_METRICS.append(metrics)
            ALL_PREDS.append(pred)

            print(
                f"[{spec['id']}] DONE | "
                f"AUROC={metrics['auroc']:.4f} | "
                f"AUPRC={metrics['auprc']:.4f} | "
                f"Sensitivity={metrics['sensitivity']:.4f} | "
                f"Specificity={metrics['specificity']:.4f} | "
                f"ECE={metrics['ece']:.4f}"
            )

            del pred
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print(f"\nCompleted fold {target}, seed {seed}.")


# ======================== 7. COMBINE EXPERIMENT RESULTS ========================

print("\nCombining all experiment results...")

summary = pd.DataFrame(ALL_METRICS)

predictions = (
    pd.concat(ALL_PREDS, ignore_index=True)
    if ALL_PREDS else pd.DataFrame()
)

STORE.save_df(summary, "ablation_summary.csv")
STORE.save_df(predictions, "all_external_predictions.csv")

# Aggregate mean and standard deviation.
aggregate_rows = []

for model_name, group in summary.groupby("model_type"):

    for metric in [
        "auroc",
        "auprc",
        "accuracy",
        "balanced_accuracy",
        "sensitivity",
        "specificity",
        "sensitivity_at_95_specificity",
        "specificity_at_90_sensitivity",
        "f1",
        "mcc",
        "brier",
        "nll",
        "ece",
        "dice_disc",
        "dice_cup",
        "model_parameters"
    ]:

        if metric not in group.columns:
            continue

        aggregate_rows.append({
            "model_type": model_name,
            "metric": metric,
            "mean": group[metric].mean(),
            "std": group[metric].std(ddof=1),
            "min": group[metric].min(),
            "max": group[metric].max(),
            "n": int(group[metric].notna().sum())
        })

aggregate = pd.DataFrame(aggregate_rows)

STORE.save_df(aggregate, "aggregate_metrics.csv")


# ======================== 8. PAIRED ABLATION COMPARISONS ========================

print("Calculating paired AUROC differences...")

PAIR_ORDER = [
    ("global_baseline", "B_global_segmentation"),
    ("B_global_segmentation", "C_roi_vcdr"),
    ("C_roi_vcdr", "D_sector_features"),
    ("D_sector_features", "E_graph_attention"),
    ("E_graph_attention", "F_domain_adversarial"),
    ("F_domain_adversarial", "G_prototype_alignment")
]

pair_rows = []

for target in CFG["fold_targets"]:

    for seed in CFG["seeds"]:

        subset = predictions[
            (predictions["held_out_source"] == target)
            & (predictions["seed"] == seed)
        ]

        for model_a, model_b in PAIR_ORDER:

            pa = subset[subset.model_type == model_a]
            pb = subset[subset.model_type == model_b]

            if pa.empty or pb.empty:
                continue

            delta, low, high, n = paired_bootstrap_auc_delta(
                pa,
                pb,
                seed=int(seed),
                n_boot=CFG["bootstrap_samples"]
            )

            pair_rows.append({
                "held_out_source": target,
                "seed": seed,
                "comparison": f"{model_b} - {model_a}",
                "model_a": model_a,
                "model_b": model_b,
                "delta_auroc": delta,
                "ci_low": low,
                "ci_high": high,
                "n_paired": n
            })

pairwise = pd.DataFrame(pair_rows)

STORE.save_df(
    pairwise,
    "pairwise_ablation_deltas.csv"
)


# ======================== 9. SUMMARY FIGURES ========================

print("Generating summary figures...")

mean_results = (
    summary.groupby("model_type")
    .mean(numeric_only=True)
    .reindex(MODEL_ORDER)
)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

plots = [
    ("auroc", "AUROC"),
    ("auprc", "AUPRC"),
    ("sensitivity", "Sensitivity"),
    ("specificity", "Specificity")
]

for ax, (metric, title) in zip(axes.flat, plots):

    if metric not in mean_results.columns:
        continue

    values = mean_results[metric].dropna()

    ax.bar(
        [x.replace("_", "\n") for x in values.index],
        values.values
    )

    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

STORE.save_figure(
    fig,
    "figures/ablation_classification_comparison.png"
)

plt.close(fig)


# Calibration and MCC comparison.
fig, ax = plt.subplots(figsize=(12, 6))

for metric in ["mcc", "ece"]:

    if metric in mean_results.columns:
        ax.plot(
            mean_results.index,
            mean_results[metric],
            marker="o",
            label=metric.upper()
        )

ax.set_title("Ablation: MCC and calibration error")
ax.legend()
ax.grid(alpha=0.25)
ax.tick_params(axis="x", rotation=35)

plt.tight_layout()

STORE.save_figure(
    fig,
    "figures/ablation_mcc_calibration.png"
)

plt.close(fig)


# Combined ROC and Precision-Recall plots for every fold.
for target in CFG["fold_targets"]:

    for seed in CFG["seeds"]:

        subset = predictions[
            (predictions["held_out_source"] == target)
            & (predictions["seed"] == seed)
        ]

        # ROC comparison
        fig, ax = plt.subplots(figsize=(8, 7))

        for model_name in MODEL_ORDER:

            p = subset[subset.model_type == model_name]

            if p.empty or p.label.nunique() < 2:
                continue

            fpr, tpr, _ = roc_curve(
                p.label.values,
                p.prob_calibrated.values
            )

            auc = roc_auc_score(
                p.label.values,
                p.prob_calibrated.values
            )

            ax.plot(fpr, tpr, label=f"{model_name}: {auc:.3f}")

        ax.plot([0, 1], [0, 1], "--")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"ROC comparison — {target}, seed {seed}")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.2)

        plt.tight_layout()

        STORE.save_figure(
            fig,
            f"figures/ROC_{target}_seed_{seed}.png"
        )

        plt.close(fig)

        # Precision-Recall comparison
        fig, ax = plt.subplots(figsize=(8, 7))

        for model_name in MODEL_ORDER:

            p = subset[subset.model_type == model_name]

            if p.empty or p.label.nunique() < 2:
                continue

            precision, recall, _ = precision_recall_curve(
                p.label.values,
                p.prob_calibrated.values
            )

            ap = average_precision_score(
                p.label.values,
                p.prob_calibrated.values
            )

            ax.plot(recall, precision, label=f"{model_name}: {ap:.3f}")

        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title(f"PR comparison — {target}, seed {seed}")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.2)

        plt.tight_layout()

        STORE.save_figure(
            fig,
            f"figures/PR_{target}_seed_{seed}.png"
        )

        plt.close(fig)


# ======================== 10. SCIENTIFIC RESULT REPORT ========================

report_lines = [
    "# Glaucomma A-G Ablation Report",
    "",
    f"Execution mode: {RUN_MODE}",
    f"Datasets: {', '.join(CFG['fold_targets'])}",
    f"Seeds: {CFG['seeds']}",
    "",
    "## Mean external classification results",
    "",
    summary.groupby("model_type")[
        ["auroc", "auprc", "sensitivity", "specificity", "mcc", "ece"]
    ].mean().round(4).to_string(),
    "",
    "## Paired AUROC differences",
    "",
    pairwise.round(4).to_string(index=False),
    "",
    "## Interpretation guidance",
    "",
    "- Compare A vs B to assess segmentation supervision.",
    "- Compare B vs C to assess ROI and vCDR information.",
    "- Compare C vs D to assess sector structural features.",
    "- Compare D vs E to assess graph attention.",
    "- Compare E vs F to assess domain adversarial learning.",
    "- Compare F vs G to assess prototype alignment.",
    "",
    "Positive AUROC delta means the second model performed better.",
    "A confidence interval crossing zero does not establish a reliable improvement.",
    "",
    "Do not claim statistical significance from one seed or one held-out dataset.",
    "Use all three seeds and all three held-out datasets for final claims.",
    "The paired bootstrap interval measures test-set sampling uncertainty;",
    "it does not replace variation across independent training seeds.",
    "",
    "All external test predictions were calibrated using source-validation data.",
    "No external test labels were used for threshold or hyperparameter selection.",
]

STORE.save_text(
    "\n".join(report_lines),
    "ablation_scientific_report.md"
)


# ======================== 11. FINAL CONFIGURATION AND MANIFEST ========================

STORE.save_json(
    {
        "run_mode": RUN_MODE,
        "fold_targets": CFG["fold_targets"],
        "seeds": CFG["seeds"],
        "run_variants": RUN_VARIANT_IDS,
        "experiment_version": EXPERIMENT_VERSION,
        "configuration": CFG,
        "ablation_plan": ABLATION_PLAN,
        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S")
    },
    "final_experiment_config.json"
)

manifest = []

for p in sorted(LOCAL_RUN.rglob("*")):

    if p.is_file():

        manifest.append({
            "file": str(p.relative_to(LOCAL_RUN)),
            "size_bytes": p.stat().st_size,
            "sha256": hashlib.sha256(p.read_bytes()).hexdigest()
        })

STORE.save_df(
    pd.DataFrame(manifest),
    "artifact_manifest.csv"
)

STORE.save_json(
    {
        "status": "COMPLETED",
        "run_name": CFG["run_name"],
        "mode": RUN_MODE,
        "folds": CFG["fold_targets"],
        "seeds": CFG["seeds"],
        "models": MODEL_ORDER,
        "metrics_rows": len(summary),
        "prediction_rows": len(predictions),
        "completed_at": time.strftime("%Y-%m-%d %H:%M:%S")
    },
    "RUN_COMPLETED.json"
)

# Create complete backup.
zip_path = shutil.make_archive(
    str(LOCAL_RUN),
    "zip",
    root_dir=LOCAL_RUN
)

drive_zip = DRIVE_RUN.parent / f"{CFG['run_name']}.zip"

shutil.copy2(zip_path, drive_zip)

print("\n" + "=" * 65)
print("GLAUCOMMA A-G ABLATION COMPLETED")
print("=" * 65)

print("Local results:", LOCAL_RUN)
print("Google Drive:", DRIVE_RUN)
print("ZIP backup:", drive_zip)

display(summary.round(4))
display(pairwise.round(4))

print("\nAll completed checkpoints and artifacts have been mirrored to Drive.")
'''

main_start = code.index(main_marker)

code = code[:main_start] + custom_main

# Syntax verification before the GPU experiment starts.
compile(code, "glaucomma_ablation_runner.py", "exec")

print("A-G ablation patch verification: PASSED")
print("Starting the original V4.5 pipeline with controlled ablations...")

# ======================== 12. EXECUTE ========================

try:
    exec(code, globals(), globals())

except BaseException:
    print("\nEXPERIMENT INTERRUPTED OR FAILED.")
    print("Check the traceback below.")
    print("Completed epoch checkpoints are preserved in Google Drive.")
    print("Run this same cell again with RESUME=True to continue.")
    traceback.print_exc()
    raise


Applying: runner_patch_v41.py
Applying: runner_patch_v42.py
Applying: runner_patch_v43.py
Applying: runner_patch_v43_autograd.py
Applying: runner_patch_v44_runtime.py
Applying: runner_patch_v45_masks.py
Applying: runner_patch_v45_lowlabels.py
V4.5 source integrity and original patches: PASSED
A-G ablation patch verification: PASSED
Starting the original V4.5 pipeline with controlled ablations...
Mounted at /content/drive
Resolved Colab output: /content/Glaucomma_runs/rimgraph_ablation_v1
Resolved Drive output: /content/drive/MyDrive/Glaucomma_RimGraphDG/rimgraph_ablation_v1
Drive write verification: PASSED

=== RIMGRAPH V4.4 RUNTIME ===
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB



## RimGraph-DG V4 configuration

,setting,value
0,kaggle_dataset,arnavjain1/glaucoma-datasets
1,manual_data_dir,
2,sources,"['ORIGA', 'REFUGE', 'G1020']"
3,fold_targets,['ORIGA']
4,image_size,320
5,num_workers,0
6,canonicalize_laterality,True
7,exclude_cross_source_duplicates,True
8,project_name,RimGraph_DG_V4
9,run_name,rimgraph_ablation_v1


## Downloading or locating Kaggle dataset

Using Colab cache for faster access to the 'glaucoma-datasets' dataset.
Dataset root: /kaggle/input/glaucoma-datasets


### Unlabelled images excluded safely

... (6 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

[BACKBONE] ready: convnext_tiny.fb_in22k_ft_in1k
[PREFLIGHT] full-model forward/backward PASSED | peak allocated=0.35 GB
[V4.5] RimGraph checkpoints from earlier revisions are intentionally NOT reused.

GLAUCOMMA A-G ABLATION EXPERIMENT
Run mode: pilot
Folds: ['ORIGA']
Seeds: [2029]
Resume: True

#################################################################
SEED 2029 | HELD-OUT DATASET: ORIGA
#################################################################

[A] Training or resuming global baseline...
[BASELINE REUSE] split signature mismatch; training baseline normally

------------------------------------------------------------
[B] B_global_segmentation
Global classification with disc/cup segmentation supervision
Stage: global_seg
Domain objective: False
Prototype objective: False
Signature: 51abccd078bc9119
------------------------------------------------------------
[B] DONE | AUROC=0.7731 | AUPRC=0.5414 | Sensitivity=0.7500 | Specificity=0.6452 | ECE=0.2134

-----------------

[BACKBONE] ready: convnext_tiny.fb_in22k_ft_in1k
[RIMGRAPH] epoch 9/30 start | stage=vcdr_only | batches=728
[RIMGRAPH] epoch 9 stage=vcdr_only batch 1/728
[RIMGRAPH] epoch 9 stage=vcdr_only batch 182/728
[RIMGRAPH] epoch 9 stage=vcdr_only batch 364/728
[RIMGRAPH] epoch 9 stage=vcdr_only batch 546/728
[RIMGRAPH] epoch 9 stage=vcdr_only batch 728/728
[RIMGRAPH] epoch 9 done | stage=vcdr_only loss=0.2984 auroc=0.9941 auprc=0.9942 disc_dice=0.9488 cup_dice=0.8821 n_disc=1456 n_cup=1270 proto_active=n/a
[RIMGRAPH] epoch 10/30 start | stage=vcdr_only | batches=728
[RIMGRAPH] epoch 10 stage=vcdr_only batch 1/728
[RIMGRAPH] epoch 10 stage=vcdr_only batch 182/728
[RIMGRAPH] epoch 10 stage=vcdr_only batch 364/728
[RIMGRAPH] epoch 10 stage=vcdr_only batch 546/728
[RIMGRAPH] epoch 10 stage=vcdr_only batch 728/728
[RIMGRAPH] epoch 10 done | stage=vcdr_only loss=0.2719 auroc=0.9956 auprc=0.9955 disc_dice=0.9539 cup_dice=0.8939 n_disc=1456 n_cup=1292 proto_active=n/a
[RIMGRAPH] epoch 11/30 start | s